# Download Evaluation Datasets

This notebook downloads and formats the evaluation datasets used in the think-overflow experiments.

**Code benchmarks:**
- EvalPlus (HumanEval+ and MBPP+)
- LiveCodeBench
- BigCodeBench
- CRUXEval (code understanding)
- EditBench (code repair)

**Math benchmarks:**
- GSM8K

**Reasoning benchmarks:**
- GPQA

In [12]:
import json
import re
import random
from pathlib import Path

from datasets import Dataset, concatenate_datasets, load_dataset
from huggingface_hub import hf_hub_download
from llm_cgr import save_jsonl

# this notebook lives in data/ so output dirs are relative to here
CODE_DIR = Path("./code")
CRUX_DIR = Path("./crux")
MATH_DIR = Path("./math")
REASONING_DIR = Path("./reasoning")

# create all directories
for d in [CODE_DIR, CRUX_DIR, MATH_DIR, REASONING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [13]:
# commit hashes for reproducibility
REVISIONS = {
    "humanevalplus": "d32357c",
    "mbppplus": "b2d74c9",
    "livecodebench": "0fe84c3",
    "bigcodebench": "b74c0d0",
    "gsm8k": "cc7b047",
    "gpqa": "fa6a028",
    "cruxeval": "b96af04",
    "editbench": "2bda4c5",
}

## Code Benchmarks

### EvalPlus (HumanEval+ and MBPP+)

HumanEval+ and MBPP+ extend the original benchmarks with additional test cases.

Sources:
- https://huggingface.co/datasets/evalplus/humanevalplus
- https://huggingface.co/datasets/evalplus/mbppplus

In [14]:
# load humaneval+ dataset
humaneval = load_dataset(
    "evalplus/humanevalplus",
    split="test",
    revision=REVISIONS["humanevalplus"],
)
print(f"loaded {len(humaneval)} humaneval+ problems")
print(f"columns: {humaneval.column_names}")

loaded 164 humaneval+ problems
columns: ['task_id', 'prompt', 'canonical_solution', 'entry_point', 'test']


In [15]:
# load mbpp+ dataset
mbpp = load_dataset(
    "evalplus/mbppplus",
    split="test",
    revision=REVISIONS["mbppplus"],
)
print(f"loaded {len(mbpp)} mbpp+ problems")
print(f"columns: {mbpp.column_names}")

loaded 378 mbpp+ problems
columns: ['task_id', 'code', 'prompt', 'source_file', 'test_imports', 'test_list', 'test']


In [16]:
# format for our pipeline
# humaneval+ provides: prompt (function signature + docstring), entry_point, test
# mbpp+ provides: code, prompt (description), test (no entry_point, extract from code)

# instructions for instruction-tuned models
HUMANEVAL_INSTRUCTION = "Complete the following Python function:\n\n"
MBPP_FUNCTION_NAME = "\n\nYour function should be named `{entry_point}`."

evalplus_records = []

# add humaneval+ problems
for row in humaneval:
    record = {
        "prompt": HUMANEVAL_INSTRUCTION + row["prompt"],
        "entry_point": row["entry_point"],
        "test_code": row["test"],
        "source": "humanevalplus",
    }
    evalplus_records.append(record)

# add mbpp+ problems (extract entry_point from code field)
for row in mbpp:
    # extract function name from the code solution
    code = row["code"]
    match = re.search(r"def\s+(\w+)\s*\(", code)
    entry_point = match.group(1) if match else None

    # append function name requirement to the original prompt
    prompt = row["prompt"] + MBPP_FUNCTION_NAME.format(entry_point=entry_point)

    record = {
        "prompt": prompt,
        "entry_point": entry_point,
        "test_code": row["test"],
        "source": "mbppplus",
    }
    evalplus_records.append(record)

save_jsonl(evalplus_records, str(CODE_DIR / "evalplus.jsonl"))
print(f"saved {len(evalplus_records)} records to code/evalplus.jsonl")

saved 542 records to code/evalplus.jsonl


### LiveCodeBench

LiveCodeBench is a continuously updated benchmark of competitive programming problems.

Source: https://huggingface.co/datasets/livecodebench/code_generation_lite

In [17]:
# load livecodebench dataset (test5 + test6 releases)
# note: datasets 4.0+ removed support for dataset scripts, so we download jsonl directly
# see: https://github.com/LiveCodeBench/LiveCodeBench/issues/107
lcb_datasets = []
for filename in ["test5.jsonl", "test6.jsonl"]:
    jsonl_path = hf_hub_download(
        repo_id="livecodebench/code_generation_lite",
        filename=filename,
        repo_type="dataset",
        revision=REVISIONS["livecodebench"],
    )
    lcb_datasets.append(Dataset.from_json(jsonl_path))

lcb = concatenate_datasets(lcb_datasets)
print(f"loaded {len(lcb)} problems (test5 + test6)")
print(f"columns: {lcb.column_names}")

loaded 342 problems (test5 + test6)
columns: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata']


In [18]:
# format for our pipeline
# livecodebench provides: question_content, public_test_cases (json string)
# note: problems are language-agnostic, so we add a Python instruction
PYTHON_INSTRUCTION = (
    "\n\nWrite a Python program that reads from stdin and writes to stdout."
)

lcb_records = []
for row in lcb:
    # parse test cases from json string
    public_tests = (
        json.loads(row["public_test_cases"]) if row["public_test_cases"] else []
    )

    # extract inputs and expected outputs
    inputs = [t["input"] for t in public_tests]
    expected_outputs = [t["output"] for t in public_tests]

    record = {
        "prompt": row["question_content"] + PYTHON_INSTRUCTION,
        "inputs": inputs,
        "expected_outputs": expected_outputs,
    }
    lcb_records.append(record)

save_jsonl(lcb_records, str(CODE_DIR / "livecodebench.jsonl"))
print(f"saved {len(lcb_records)} records to code/livecodebench.jsonl")

saved 342 records to code/livecodebench.jsonl


### BigCodeBench

BigCodeBench is a benchmark of 1,140 tasks that test real-world library usage (pandas, numpy, etc.). Each task uses `unittest.TestCase` for evaluation.

Source: https://huggingface.co/datasets/bigcode/bigcodebench

In [19]:
# load bigcodebench from huggingface — avoids the bigcodebench package which has
# broken transitive dependencies (requires wget, which is not installed)
bcb = load_dataset(
    "bigcode/bigcodebench",
    split="v0.1.2",
    revision=REVISIONS["bigcodebench"],
)
print(f"loaded {len(bcb)} bigcodebench problems")
print(f"columns: {bcb.column_names}")

# format for our pipeline
# bigcodebench provides: task_id, instruct_prompt (natural language instruction),
# entry_point (always "task_func"), test (unittest.TestCase class)
bcb_records = [
    {
        "task_id": row["task_id"],
        "prompt": row["instruct_prompt"],
        "entry_point": row["entry_point"],  # always "task_func"
        "test_code": row["test"],  # unittest.TestCase class
    }
    for row in bcb
]

save_jsonl(bcb_records, str(CODE_DIR / "bigcodebench.jsonl"))
print(f"saved {len(bcb_records)} records to code/bigcodebench.jsonl")

loaded 1140 bigcodebench problems
columns: ['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs']
saved 1140 records to code/bigcodebench.jsonl


### EditBench

EditBench is a benchmark of real-world code editing tasks drawn from GitHub issues.
Each task provides original code, a highlighted section to edit, and an instruction.
Tests are pytest-based and validate the complete modified file.

We include only Python tasks (JavaScript evaluation requires a JS runtime).
We use the `test` split (held-out problems not seen during model training).

Source: https://huggingface.co/datasets/copilot-arena/editbench

In [20]:
# load editbench test split
editbench = load_dataset(
    "copilot-arena/editbench",
    split="complete",
    revision=REVISIONS["editbench"],  # None = latest main; pin after first run
)
print(f"loaded {len(editbench)} tasks from editbench test split")
print(f"columns: {editbench}")

loaded 540 tasks from editbench test split
columns: Dataset({
    features: ['problem_id', 'pair_id', 'programming_language', 'natural_language', 'cursor_position', 'python_version', 'original_code', 'highlighted_code', 'instruction', 'test_code', 'requirements', 'test_harness', 'split'],
    num_rows: 540
})


In [21]:
# format for our pipeline
# editbench provides: instruction (editing task), original_code (full file),
# highlighted_code (section to change), test_code (pytest tests), test_harness
# (dict of auxiliary test files: conftest.py, test_utils.py, etc.)
#
# prompt: ask the model to output the complete modified file
# evaluation: write the model output to a temp dir and run pytest

EDITBENCH_PROMPT = (
    "You are given a Python file and an instruction to edit it. "
    "Apply the instruction to the highlighted section and "
    "output the complete modified file.\n\n"
    "Instruction: {instruction}\n\n"
    "Current file:\n```python\n{original_code}\n```\n\n"
    "Section to edit:\n```python\n{highlighted_code}\n```"
)

editbench_records = []
for row in editbench:
    record = {
        "prompt": EDITBENCH_PROMPT.format(
            instruction=row["instruction"],
            original_code=row["original_code"],
            highlighted_code=row["highlighted_code"],
        ),
        "test_code": row["test_code"],
        # auxiliary test files (conftest.py, test_utils.py, etc.) used by pytest
        "test_harness": row["test_harness"],
        "requirements": row["requirements"],
    }
    editbench_records.append(record)

save_jsonl(editbench_records, str(CODE_DIR / "editbench.jsonl"))
print(f"saved {len(editbench_records)} records to code/editbench.jsonl")

saved 540 records to code/editbench.jsonl


### CRUXEval (Code Understanding)

CRUXEval tests code *understanding* rather than generation: given a Python function,
the model either predicts the output for a given input (CRUXEval-O) or finds an input
that produces a given output (CRUXEval-I).

- https://huggingface.co/datasets/cruxeval-org/cruxeval

In [22]:
# load cruxeval dataset (single test split, 800 problems)
cruxeval = load_dataset(
    "cruxeval-org/cruxeval",
    split="test",
    revision=REVISIONS["cruxeval"],
)
print(f"loaded {len(cruxeval)} problems")

loaded 800 problems


In [23]:
# format for our pipeline
# cruxeval provides: code (function), input, output, id
# we create two separate datasets for the two tasks:
#   cruxeval_o: predict output given code + input
#   cruxeval_i: predict input given code + expected output

# --- cruxeval-o: output prediction ---
# prompt matches the original paper's direct (non-cot) format:
#   https://github.com/facebookresearch/cruxeval/blob/main/prompts.py
CRUXEVAL_O_PROMPT = (
    """You are given a Python function and an assertion containing an input to the function. """
    """Complete the assertion with a literal (no unsimplified expressions, no function calls) """
    """containing the output when executing the provided code on the given input, even if the """
    """function is incorrect or incomplete. Do NOT output any extra information. Provide the """
    """full assertion with the correct output in [ANSWER] and [/ANSWER] tags.
"""
    """
Here are 2 examples, showing the expected format.
"""
    """
[PYTHON]
def f(n):
    return n
assert f(17) == ??
[/PYTHON]
"""
    """[ANSWER]
assert f(17) == 17
[/ANSWER]
"""
    """
[PYTHON]
def f(s):
    return s + "a"
assert f("x9j") == ??
[/PYTHON]
"""
    """[ANSWER]
assert f("x9j") == "x9ja"
[/ANSWER]
"""
    """
Now solve the following problem:
"""
    """
[PYTHON]
{code}
assert f({input}) == ??
[/PYTHON]
"""
)

cruxeval_o_records = []
for row in cruxeval:
    # fill in the code and input for this problem
    prompt = CRUXEVAL_O_PROMPT.format(code=row["code"], input=row["input"])
    cruxeval_o_records.append(
        {
            "prompt": prompt,
            "answer": row["output"],  # ground truth output for evaluation
            "task": "output",
        }
    )

save_jsonl(cruxeval_o_records, str(CRUX_DIR / "cruxeval_o.jsonl"))
print(f"saved {len(cruxeval_o_records)} records to crux/cruxeval_o.jsonl")

# --- cruxeval-i: input prediction ---
CRUXEVAL_I_PROMPT = (
    """You will be given a function f and an output in the form f(??) == output. Find any """
    """input such that executing f on the input leads to the given output. There may be """
    """multiple answers, but you should only output one. In [ANSWER] and [/ANSWER] tags, """
    """complete the assertion with one such input that will produce the output when executing """
    """the function.
"""
    """
Here are 2 examples, showing the expected format.
"""
    """
[PYTHON]
def f(my_list):
    count = 0
    for i in my_list:
        if len(i) % 2 == 0:
            count += 1
    return count
assert f(??) == 3
[/PYTHON]
"""
    """[ANSWER]
assert f(["mq", "px", "zy"]) == 3
[/ANSWER]
"""
    """
[PYTHON]
def f(s1, s2):
    return s1 + s2
assert f(??) == "banana"
[/PYTHON]
"""
    """[ANSWER]
assert f("ba", "nana") == "banana"
[/ANSWER]
"""
    """
Now solve the following problem:
"""
    """
[PYTHON]
{code}
assert f(??) == {output}
[/PYTHON]
"""
)

cruxeval_i_records = []
for row in cruxeval:
    # fill in the code and expected output for this problem
    prompt = CRUXEVAL_I_PROMPT.format(code=row["code"], output=row["output"])
    cruxeval_i_records.append(
        {
            "prompt": prompt,
            "code": row["code"],  # needed to execute f(predicted) for evaluation
            "answer": row[
                "output"
            ],  # expected output used to verify the predicted input
            "task": "input",
        }
    )

save_jsonl(cruxeval_i_records, str(CRUX_DIR / "cruxeval_i.jsonl"))
print(f"saved {len(cruxeval_i_records)} records to crux/cruxeval_i.jsonl")

saved 800 records to crux/cruxeval_o.jsonl
saved 800 records to crux/cruxeval_i.jsonl


## Math Benchmarks

### GSM8K

Grade School Math 8K — a dataset of 8.5K grade school math word problems.

Source: https://huggingface.co/datasets/openai/gsm8k

In [24]:
# load gsm8k dataset (test split)
gsm8k = load_dataset(
    "openai/gsm8k",
    "main",
    split="test",
    revision=REVISIONS["gsm8k"],
)
print(f"loaded {len(gsm8k)} problems")

# format for our pipeline
# gsm8k provides: question, answer (with #### final_answer format)
gsm8k_records = []
for row in gsm8k:
    # extract the final numeric answer after ####
    answer_text = row["answer"]
    match = re.search(r"####\s*(.+)$", answer_text)
    final_answer = match.group(1).strip() if match else answer_text

    record = {
        "prompt": row["question"],
        "answer": final_answer,
    }
    gsm8k_records.append(record)

save_jsonl(gsm8k_records, str(MATH_DIR / "gsm8k.jsonl"))
print(f"saved {len(gsm8k_records)} records to math/gsm8k.jsonl")

loaded 1319 problems
saved 1319 records to math/gsm8k.jsonl


## Reasoning Benchmarks

### GPQA (Graduate-Level Google-Proof Q&A)

A multiple-choice Q&A dataset of very hard questions written and validated by experts in biology, physics, and chemistry.

Source: https://huggingface.co/datasets/Idavidrein/gpqa

In [25]:
# load gpqa dataset (main subset)
# note: gpqa only has a "train" split (the dataset is designed as evaluation data)
gpqa = load_dataset(
    "Idavidrein/gpqa",
    "gpqa_main",
    split="train",
    revision=REVISIONS["gpqa"],
)
print(f"loaded {len(gpqa)} problems")

# format for our pipeline
# gpqa provides: Question, Correct Answer, Incorrect Answer 1/2/3
gpqa_records = []
for idx, row in enumerate(gpqa):
    # collect all choices and shuffle them
    correct = row["Correct Answer"]
    incorrect = [
        row["Incorrect Answer 1"],
        row["Incorrect Answer 2"],
        row["Incorrect Answer 3"],
    ]

    # create shuffled choices with tracked correct index
    choices = [correct] + incorrect
    indices = list(range(4))
    random.seed(idx)  # deterministic per question (stable across python versions)
    random.shuffle(indices)
    shuffled_choices = [choices[i] for i in indices]
    correct_index = indices.index(0)  # where did the correct answer end up?

    # format prompt with question and choices
    choices_text = "\n".join(
        [f"{chr(65 + i)}. {c}" for i, c in enumerate(shuffled_choices)]
    )
    prompt = f"{row['Question']}\n\n{choices_text}"

    # the answer is the letter corresponding to the correct choice
    answer = chr(65 + correct_index)

    record = {
        "prompt": prompt,
        "answer": answer,
    }
    gpqa_records.append(record)

save_jsonl(gpqa_records, str(REASONING_DIR / "gpqa.jsonl"))
print(f"saved {len(gpqa_records)} records to reasoning/gpqa.jsonl")

loaded 448 problems
saved 448 records to reasoning/gpqa.jsonl


## Summary

All datasets have been downloaded and formatted. Run inference with:

```bash
# baseline (single-pass) inference
infer -m qwen3-8b -d code/evalplus --baseline
infer -m qwen3-8b -d math/gsm8k --baseline
infer -m qwen3-8b -d reasoning/gpqa --baseline

# two-pass overflow inference
infer -m qwen3-8b -d code/evalplus --max-think-tokens 8192 --overflow-suffix formal
```

In [26]:
# list all downloaded datasets
print("downloaded datasets:")
for d in [CODE_DIR, CRUX_DIR, MATH_DIR, REASONING_DIR]:
    for f in sorted(d.glob("*.jsonl")):
        with open(f) as fp:
            count = sum(1 for _ in fp)
        print(f"  {f}: {count} records")

downloaded datasets:
  code/bigcodebench.jsonl: 1140 records
  code/editbench.jsonl: 540 records
  code/evalplus.jsonl: 542 records
  code/livecodebench.jsonl: 342 records
  crux/cruxeval_i.jsonl: 800 records
  crux/cruxeval_o.jsonl: 800 records
  math/gsm8k.jsonl: 1319 records
  reasoning/gpqa.jsonl: 448 records
